In [ ]:
# ADC設計メモ

3章まで読んだが、イチから全部丸々読んでいたら日が暮れてしまうのでΔΣから学習。 分からないところがあれば都度戻るようにする。

---

## 1次ΔΣADC

### 1bit ΔΣADC回路

教科書に掲載されている1次1bit ΔΣADC回路。

![1次理想ADC](./image/ideal_DeltaSigma.png "ideal_DeltaSigma")

---

## 理想素子による実装

Cadence Virtuoso上で理想素子を用いて1次ΔΣADCを構成した。

![1次ADC](./image/DeltaSigma.png "DeltaSigma")

### 構成

- スイッチトキャパシタ積分器
- 1bitコンパレータ
- DFFによる離散時間量子化器
- 1bit DACフィードバック

### 設計時のポイント

- コンパレータは連続時間ではなくDFFを用いて離散時間化
- フィードバックDACの極性が負帰還となるように接続
- 入力振幅はDAC基準電圧範囲内に設定

---

## 過渡応答確認

### シミュレーション条件

| Parameter | Value |
|---|---|
| Input Frequency | 100 kHz |
| Input Amplitude | 0.4 V |
| Clock Frequency | 100 MHz |

### 出力波形

![出力波形](./image/VDAC.png "VDAC")

入力信号に応じてパルス密度が変化していることを確認した。
入力電圧が高い領域ではHighが多くなり、入力電圧が低い領域ではLowが多くなる。
1bit ΔΣADCにおける典型的な粗密波が得られた。

---

## FFTによるノイズシェーピング確認

入力信号を加えず、微小DCオフセットを与えた状態でFFTを実施した。

### FFT条件

| Parameter | Value |
|---|---|
| FFT Points | 65536 |
| Sampling Clock | 100 MHz |
| Observation Time | 655.36 µs |

### ノイズシェーピング特性

![ノイズシェーピング特性](./image/noiseshaping_noInput.png "noiseshaping_noInput")

## スペクトル表現の違い

FFT/DFTの結果は目的に応じて様々な形で表現される。

| 名称 | 数式 | 単位 | 意味 | FFT点数依存 |
|---|---|---|---|---|
| DFT振幅スペクトル | $|X[k]|$ | DFT係数 | DFTで得られた複素係数の絶対値 | あり |
| 振幅スペクトル (Amplitude Spectrum) | $|X(f)|$ | V | 各周波数成分の振幅 | あり |
| パワースペクトル (Power Spectrum) | $|X(f)|^2$ | V² | 各周波数成分が持つパワー | あり |
| 振幅スペクトル密度 (ASD) | $\frac{|X(f)|}{\sqrt{\Delta f}}$ | V/√Hz | 1√Hzあたりの振幅 | なし |
| パワースペクトル密度 (PSD) | $\frac{|X(f)|^2}{\Delta f}$ | V²/Hz | 1Hzあたりのパワー | なし |

ここで、

$$\Delta f = \frac{f_s}{N}$$

- $f_s$：サンプリング周波数
- $N$：FFTポイント数

である。

---

## dB表示した場合

| 名称 | dB表記 |
|---|---|
| 振幅スペクトル | $20\log_{10}|X|$ |
| パワースペクトル | $10\log_{10}|X|^2$ |
| ASD | $20\log_{10}\left(\frac{|X|}{\sqrt{\Delta f}}\right)$ |
| PSD | $10\log_{10}\left(\frac{|X|^2}{\Delta f}\right)$ |

振幅スペクトルとパワースペクトルは

$$20\log_{10}|X| = 10\log_{10}|X|^2$$

となるため、dB表示では同じ値になる。

---

## イメージ

```text
時間波形 x[n]
      │
      ▼
DFT/FFT
      │
      ▼
DFT振幅スペクトル
|X[k]|
      │
      ├─ 振幅として見る
      ▼
振幅スペクトル
|X(f)|
      │
      ├─ 二乗
      ▼
パワースペクトル
|X(f)|²
      │
      ├─ 周波数分解能 Δf で割る
      ▼
PSD
|X(f)|² / Δf
      │
      └─ 平方根
      ▼
ASD
|X(f)| / √Δf
```

## 「FFT点数とノイズフロア」および「kT/Cノイズのサンプリング周波数依存性」の関係

### 1. FFT点数とノイズフロア（信号処理上の変化）

* **アクション:** FFTの点数 ($N$) を増やす
* **メカニズム:** 周波数分解能が細かくなり、ノイズがより多くのビンに分割される
* **結果:** 1ビン当たりのエネルギーが低下し、**見かけ上のノイズフロアが減少したように見える**

### 2. kT/Cノイズとサンプリング周波数（物理的な分散）

* **アクション:** サンプリング周波数 ($f_s$) を高くする
* **メカニズム:** 積分器などの熱雑音 (kT/C) が、より広いナイキスト帯域へ引き伸ばされて分散する
* **結果:** 1Hz当たりのノイズ密度が薄まり、特定帯域内の**見かけ上のノイズが減少したように見える**
